# 🌐 Federated Learning Training — Flower (FedAvg Simulation)
**Privacy-Preserving Cancer Detection Project**

This notebook uses the **Flower (`flwr`)** framework to simulate
Federated Learning across 3 hospital nodes using the `FedAvg` strategy.

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. Upload these to **MyDrive/FL_Project/** on Google Drive:
   - `ml/` folder (model.py)
   - `data/` folder (node1/, node2/, node3/, test/)
3. Run all cells **top to bottom** (Runtime → Run all)

---

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅  Drive mounted.")

## Step 2 — Install Flower

In [ ]:
!pip install -q flwr[simulation]

import flwr
print(f"✅  Flower version: {flwr.__version__}")

## Step 3 — Copy files from Drive to Colab VM

In [ ]:
import shutil, os

PROJECT_VM   = "/content/FL_Project"
DRIVE_PATH   = "/content/drive/MyDrive/FL_Project"

os.makedirs(PROJECT_VM, exist_ok=True)

for folder in ["ml", "data"]:
    src = os.path.join(DRIVE_PATH, folder)
    dst = os.path.join(PROJECT_VM, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✅  Copied {folder}/")
    else:
        print(f"  ❌  {folder}/ NOT FOUND on Drive!")

os.makedirs(f"{PROJECT_VM}/models", exist_ok=True)
os.makedirs(f"{PROJECT_VM}/results", exist_ok=True)
print()
print("✅  All files copied to Colab VM.")

## Step 4 — Verify all required files

In [ ]:
import os, numpy as np

P = "/content/FL_Project"

required = [
    "ml/model.py",
    "data/node1/X.npy", "data/node1/y.npy",
    "data/node2/X.npy", "data/node2/y.npy",
    "data/node3/X.npy", "data/node3/y.npy",
    "data/test/X.npy",  "data/test/y.npy",
]

all_ok = True
for f in required:
    path = os.path.join(P, f)
    exists = os.path.exists(path)
    if exists and f.endswith(".npy"):
        arr = np.load(path)
        size = os.path.getsize(path) / (1024**2)
        print(f"  ✅  {f}  shape={arr.shape}  ({size:.1f} MB)")
    elif exists:
        print(f"  ✅  {f}")
    else:
        print(f"  ❌  {f}  MISSING")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Upload missing files to MyDrive/FL_Project/")
print("\n✅  All files present. Ready to train!")

## Step 5 — Define Flower Client (runs on each hospital node)

Each `FlowerClient` loads its own local data partition,
trains the model locally, and returns the updated weights to the server.
The clients **never share raw data** — only model parameters.

In [ ]:
import sys, gc, time, json
import numpy as np
import tensorflow as tf
import flwr as fl
from flwr.common import ndarrays_to_parameters, parameters_to_ndarrays

# GPU memory growth
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPUs: {len(gpus)}  (memory growth enabled)")

# Config
PROJECT      = "/content/FL_Project"
DATA_DIR     = f"{PROJECT}/data"
MODEL_DIR    = f"{PROJECT}/models"
RESULTS_DIR  = f"{PROJECT}/results"

NUM_ROUNDS     = 10
LOCAL_EPOCHS   = 3
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
NUM_NODES      = 3

# Import model
sys.path.insert(0, f"{PROJECT}/ml")
from model import build_cnn

# Load test data (shared across all clients for evaluation)
X_test = np.load(f"{DATA_DIR}/test/X.npy")
y_test = np.load(f"{DATA_DIR}/test/y.npy")


def load_node(node_id):
    """Load data for a specific hospital node."""
    X = np.load(f"{DATA_DIR}/node{node_id}/X.npy")
    y = np.load(f"{DATA_DIR}/node{node_id}/y.npy")
    return X, y


class FlowerClient(fl.client.NumPyClient):
    """Flower client representing one hospital node."""

    def __init__(self, node_id):
        self.node_id = node_id
        self.x_train, self.y_train = load_node(node_id)
        print(f"  [Client {node_id}] Loaded {len(self.x_train)} samples")

    def _build_model(self):
        """Build a fresh model to avoid GPU memory leaks."""
        tf.keras.backend.clear_session()
        model = build_cnn()
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        return model

    def get_parameters(self, config):
        """Return current model parameters."""
        model = self._build_model()
        return model.get_weights()

    def fit(self, parameters, config):
        """Train locally and return updated weights."""
        model = self._build_model()
        model.set_weights(parameters)

        history = model.fit(
            self.x_train, self.y_train,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.1,
            verbose=0,
        )

        train_loss = history.history["loss"][-1]
        train_acc  = history.history["accuracy"][-1]
        print(f"  [Client {self.node_id}] loss={train_loss:.4f}  acc={train_acc:.4f}")

        return model.get_weights(), len(self.x_train), {
            "train_loss": float(train_loss),
            "train_accuracy": float(train_acc),
        }

    def evaluate(self, parameters, config):
        """Evaluate on the shared test set."""
        model = self._build_model()
        model.set_weights(parameters)
        loss, acc = model.evaluate(X_test, y_test, verbose=0)
        print(f"  [Client {self.node_id}] eval_loss={loss:.4f}  eval_acc={acc:.4f}")
        return float(loss), len(X_test), {"eval_accuracy": float(acc)}


def client_fn(cid: str):
    """Factory function: Flower calls this to create each client."""
    node_id = int(cid) + 1  # cid 0->node1, 1->node2, 2->node3
    return FlowerClient(node_id).to_client()


print("✅  Flower client defined.")

## Step 6 — Run Flower Simulation (10 rounds, FedAvg, 3 nodes)

Flower's `start_simulation` orchestrates everything:
- Creates 3 virtual clients (one per hospital)
- Each round, selects clients, sends global weights, collects updates
- Applies **FedAvg** to aggregate weights
- Evaluates the global model after each round

In [ ]:
from flwr.server.strategy import FedAvg

# Build initial model to get starting parameters
tf.keras.backend.clear_session()
gc.collect()
init_model = build_cnn()
init_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
init_params = ndarrays_to_parameters(init_model.get_weights())

init_loss, init_acc = init_model.evaluate(X_test, y_test, verbose=0)
print(f"Initial accuracy (random): {init_acc:.4f}")
print()

# Define FedAvg strategy
strategy = FedAvg(
    fraction_fit=0.67,           # Train on ~2/3 of clients each round
    fraction_evaluate=1.0,       # Evaluate on all clients
    min_fit_clients=2,           # At least 2 clients per round
    min_evaluate_clients=3,      # Evaluate on all 3
    min_available_clients=3,     # Wait for all 3 clients
    initial_parameters=init_params,
)

print("\n" + "═" * 60)
print("  Federated Learning — Cancer Detection (Flower)")
print("  Strategy: FedAvg | Rounds: 10 | Nodes: 3 | Local epochs: 3")
print("═" * 60 + "\n")

t0 = time.time()

# Run the simulation
history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_NODES,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy,
    client_resources={"num_cpus": 1, "num_gpus": 0.3},
)

total_time = time.time() - t0

print()
print("═" * 60)
print(f"  ✅  FL Training Complete!  ({total_time:.1f}s)")
print("═" * 60)

## Step 7 — Extract metrics & save model

In [ ]:
import json, os

# Extract round-by-round metrics from Flower history
round_metrics = []

# Distributed evaluate metrics (from client-side evaluate)
eval_accs = dict(history.metrics_distributed.get("eval_accuracy", []))

# Centralized losses (from server-side)
losses = dict(history.losses_distributed)

for r in range(1, NUM_ROUNDS + 1):
    round_metrics.append({
        "round": r,
        "test_accuracy": round(float(eval_accs.get(r, 0)), 4),
        "test_loss": round(float(losses.get(r, 0)), 4),
    })

final_acc  = round_metrics[-1]["test_accuracy"]
final_loss = round_metrics[-1]["test_loss"]

# Save the final global model
# Re-run one more aggregation to get final weights
tf.keras.backend.clear_session()
final_model = build_cnn()
final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Get final parameters from strategy
final_params = strategy.parameters
if final_params is not None:
    final_weights = parameters_to_ndarrays(final_params)
    final_model.set_weights(final_weights)

# Save model
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = f"{MODEL_DIR}/federated_model.h5"
final_model.save(model_path)
size = os.path.getsize(model_path) / (1024**2)
print(f"✅  Model saved: {model_path}  ({size:.1f} MB)")

# Save metrics
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics = {
    "framework": "Flower (flwr)",
    "strategy": "FedAvg",
    "num_rounds": NUM_ROUNDS,
    "num_nodes": NUM_NODES,
    "local_epochs": LOCAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "initial_test_accuracy": round(float(init_acc), 4),
    "final_test_accuracy": final_acc,
    "final_test_loss": final_loss,
    "total_training_time_sec": round(total_time, 1),
    "round_metrics": round_metrics,
}
metrics_path = f"{RESULTS_DIR}/fl_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"✅  Metrics saved: {metrics_path}")

## Step 8 — Copy model + metrics back to Google Drive

In [ ]:
import shutil, os

DRIVE_OUT = "/content/drive/MyDrive/FL_Project"

saves = {
    f"{MODEL_DIR}/federated_model.h5":   f"{DRIVE_OUT}/models/federated_model.h5",
    f"{RESULTS_DIR}/fl_metrics.json":     f"{DRIVE_OUT}/results/fl_metrics.json",
}

for src, dst in saves.items():
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size = os.path.getsize(src) / (1024**2)
        print(f"  ✅  {os.path.basename(src)}  ({size:.2f} MB) → Drive")
    else:
        print(f"  ❌  {src} not found")

print()
print("✅  Done! federated_model.h5 is on your Google Drive.")
print(f"   Location: {DRIVE_OUT}/models/")
print()
print("Next: download federated_model.h5 from Drive to your local")
print("      models/ folder to complete the project.")

## Step 9 — Display round-by-round metrics

In [ ]:
print("═" * 55)
print("  Round-by-Round FL Metrics (Flower FedAvg)")
print("═" * 55)
print(f"  {'Round':<7} {'Test Acc':<12} {'Test Loss':<12}")
print("─" * 55)
for rm in round_metrics:
    print(f"  {rm['round']:<7} {rm['test_accuracy']:<12.4f} {rm['test_loss']:<12.4f}")
print("─" * 55)
print(f"  Final accuracy: {final_acc:.4f}")
print(f"  Total time:     {total_time:.1f}s")
print(f"  Framework:      Flower (flwr)")